# 01. LangGraph 소개 — 상태 기반 에이전트 오케스트레이션 프레임워크

## 학습 목표

LangGraph의 핵심 개념과 두 가지 API(Graph API, Functional API)를 이해합니다.

## 1.1 LangGraph란?

LangGraph는 LangChain 생태계의 **저수준 오케스트레이션 프레임워크**입니다.

### LangChain 3계층 구조

| 계층 | 역할 | 설명 |
|------|------|------|
| Deep Agents | 고수준 | 사전 구축된 에이전트 시스템 |
| LangChain | 에이전트 | LLM 에이전트 구축 도구 |
| **LangGraph** | **워크플로** | **상태 기반 오케스트레이션** |

### 핵심 특징

- **상태 관리**: TypedDict 기반 상태 정의 및 리듀서
- **지속성**: 체크포인터를 통한 상태 자동 저장
- **스트리밍**: 실시간 토큰 단위 출력
- **Human-in-the-loop**: 사람의 개입을 위한 중단/재개
- **내구성 실행**: 장애 발생 시 자동 복구

## 1.2 핵심 개념

LangGraph는 **그래프 구조**를 기반으로 워크플로를 정의합니다.

### 구성 요소

| 개념 | 설명 |
|------|------|
| **노드(Node)** | 처리 단위 — Python 함수로 정의 |
| **엣지(Edge)** | 노드 간 연결, 조건부 분기 가능 |
| **상태(State)** | TypedDict로 정의, 노드 간 공유 데이터 |
| **체크포인터(Checkpointer)** | 각 단계 상태 자동 저장 |

### 그래프 구조 다이어그램

![StateGraph 구조 — START→NodeA→조건분기→END](../assets/images/stategraph_structure.png)

## 1.3 두 가지 API

LangGraph는 동일한 기능을 두 가지 다른 스타일로 구현할 수 있는 API를 제공합니다.

| 특성 | Graph API | Functional API |
|------|----------|----------------|
| 접근 방식 | 선언적 (노드+엣지) | 명령적 (Python 제어 흐름) |
| 상태 관리 | 명시적 State + 리듀서 | 함수 스코프, 리듀서 불필요 |
| 시각화 | 그래프 시각화 지원 | 미지원 |
| 체크포인팅 | 슈퍼스텝마다 새 체크포인트 | 태스크별, 기존 체크포인트에 저장 |
| 적합 상황 | 복잡한 워크플로, 팀 개발 | 기존 코드 마이그레이션, 간단한 흐름 |

## 1.4 환경 설정 및 설치 확인

필요한 패키지가 올바르게 설치되어 있는지 확인합니다.

In [1]:
import importlib

packages = {
    "langgraph": "langgraph",
    "langchain": "langchain",
    "langchain_openai": "langchain-openai",
}

print("=" * 50)
print("LangGraph 환경 확인")
print("=" * 50)

for module_name, package_name in packages.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "installed")
        print(f"  OK  {package_name}: {version}")
    except ImportError:
        print(f"  ERR {package_name}: 설치되지 않음")

LangGraph 환경 확인
  OK  langgraph: installed
  OK  langchain: 1.2.10


  OK  langchain-openai: installed


In [2]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

required = ["OPENAI_API_KEY"]
optional = ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]

print("API 키 상태:")
for key in required:
    print(f"  {'OK' if os.environ.get(key) else 'MISSING'} {key} (필수)")
for key in optional:
    print(f"  {'OK' if os.environ.get(key) else '--'} {key} (선택)")

API 키 상태:
  OK OPENAI_API_KEY (필수)
  OK TAVILY_API_KEY (선택)
  -- LANGSMITH_API_KEY (선택)


In [3]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


In [4]:
# Core import verification
from langgraph.graph import StateGraph, START, END
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI

print("모든 핵심 임포트 완료")

모든 핵심 임포트 완료


## 1.5 Graph API 맛보기

Graph API는 **선언적** 방식으로 워크플로를 정의합니다.

1. `StateGraph(State)` — 상태 스키마로 그래프 빌더 생성
2. `add_node()` — 노드(함수) 등록
3. `add_edge()` — 노드 간 연결
4. `compile()` — 실행 가능한 그래프 생성
5. `invoke()` — 그래프 실행

In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SimpleState(TypedDict):
    input: str
    output: str

def greet(state: SimpleState) -> dict:
    return {"output": f"안녕하세요, {state['input']}님!"}

builder = StateGraph(SimpleState)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

graph = builder.compile()
result = graph.invoke({"input": "LangGraph"}, config=lf_config)
print("Graph API 결과:", result["output"])

Graph API 결과: 안녕하세요, LangGraph님!


## 1.6 Functional API 맛보기

Functional API는 **명령적** 방식으로 워크플로를 정의합니다.

- `@task` — 단위 작업 정의 (체크포인팅 단위)
- `@entrypoint` — 워크플로 진입점 정의
- 일반 Python 제어 흐름(`if`, `for`, `while` 등)을 그대로 사용

In [6]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

@task
def process(text: str) -> str:
    return f"처리 완료: {text.upper()}"

@entrypoint(checkpointer=InMemorySaver())
def simple_workflow(text: str) -> str:
    result = process(text).result()
    return result

output = simple_workflow.invoke("안녕하세요 langgraph", {**{"configurable": {"thread_id": "demo"}}, **lf_config})
print("Functional API 결과:", output)

Functional API 결과: 처리 완료: 안녕하세요 LANGGRAPH


## 1.7 다음 단계

다음 노트북에서 Graph API를 본격적으로 학습합니다.

- **02_graph_api.ipynb** — StateGraph, 노드, 엣지, 조건부 분기, 상태 리듀서

## 요약

| 항목 | 설명 |
|------|------|
| LangGraph | 상태 기반 에이전트 오케스트레이션 프레임워크 |
| Graph API | `StateGraph`로 명시적 상태 흐름 정의 |
| Functional API | `@entrypoint` + `@task`로 함수형 워크플로 |
| 핵심 개념 | State (상태), Node (노드), Edge (엣지) |
| 체크포인터 | 상태 지속성, 멀티턴 대화, 타임 트래블 지원 |

### 다음 단계
→ **[02_graph_api.ipynb](./02_graph_api.ipynb)**: Graph API의 핵심 개념을 배웁니다.
